# RAG Architecture: Understanding the Complete RAG Pipeline

> **A practical introduction to the complete Retrieval-Augmented Generation pipeline.**

Large language models are remarkably capable, but they don't automatically know everything an application needs them to know. Private company documents, newly published information, internal policies, and other domain-specific knowledge may not be available in the model's parameters.

Retrieval-Augmented Generation (RAG) addresses this by retrieving relevant information from an external knowledge source and providing it to an LLM as context at inference time.

In this tutorial, we'll look inside a RAG system and follow information from its original document all the way to the final answer generated by an LLM.

## What we'll learn

By the end of this tutorial, you'll understand:

- Why RAG is useful
- The difference between the indexing and query pipelines
- What ingestion, chunking, embeddings, indexing, retrieval, reranking, and generation do
- How retrieved information becomes context for an LLM
- Why provenance and citations matter
- Where a RAG system can fail
- How the simple RAG architecture evolves toward a production system

**This notebook is conceptual.** We will implement the individual components in later notebooks.

## 1. What Is Retrieval-Augmented Generation?

A Large Language Model (LLM) has knowledge encoded in its learned parameters. However, many applications need the model to work with information that is private, recent, domain-specific, or frequently changing.

RAG solves this by retrieving relevant external information and supplying it to the LLM as context.

```text
User Query
     +
Retrieved Information
     ↓
    LLM
     ↓
  Answer
```

The key idea is that the model does not need to contain all of the application's knowledge in its parameters. Relevant external information can be retrieved at inference time.

---
## 2. The Complete RAG Architecture

A RAG system can be viewed as two major workflows:

1. **Indexing pipeline** — prepares external knowledge for retrieval.
2. **Query pipeline** — retrieves and uses that knowledge to answer a user's question.

```text
                         RAG SYSTEM
                             │
              ┌──────────────┴──────────────┐
              │                             │
              ▼                             ▼
       INDEXING PIPELINE              QUERY PIPELINE
              │                             │
          Documents                       Query
              │                             │
              ▼                             ▼
         Ingestion                  Query Processing
              │                             │
              ▼                             ▼
          Chunking                     Retrieval
              │                             │
              ▼                             ▼
         Embeddings                    Reranking
              │                             │
              ▼                             ▼
          Indexing                  Context Assembly
              │                             │
              ▼                             ▼
       Knowledge Base                      LLM
                                            │
                                            ▼
                                         Answer
```

We will progressively implement and evaluate these components throughout the course.

---
# Part I — Preparing the Knowledge

Before a user asks a question, documents need to be transformed into a representation that can be searched efficiently.

---
## 3. Document Ingestion

Everything starts with documents. They might come from PDFs, Word documents, HTML pages, Markdown files, emails, databases, images, or scanned documents.

We cannot simply throw these files into a vector database. First, we need to extract their content and, where possible, preserve useful structure and metadata.

```text
company-policy.pdf
        ↓
     Parser
        ↓
Text + Structure + Metadata
```

A document can contain headings, paragraphs, tables, images, lists, page information, and other metadata. Losing this information can affect downstream retrieval.

We will explore document ingestion in detail in **Module 3**.

---
## 4. Chunking

Large documents are normally divided into smaller units called **chunks**.

```text
Document
│
├── Chunk 1
├── Chunk 2
├── Chunk 3
├── Chunk 4
└── ...
```

The goal is not simply to split text into arbitrary pieces. A useful chunk should contain enough information to be meaningful during retrieval while avoiding unnecessary noise.

This creates important engineering questions:

- How large should a chunk be?
- Where should it be split?
- Should headings remain attached to their paragraphs?
- How should tables be handled?
- Should chunks overlap?

We will investigate these questions experimentally in **Module 4 — Chunking**.

---
## 5. Embeddings

An embedding model converts text into a numerical vector.

```text
"Customers can request a refund within 30 days."
                       ↓
                Embedding Model
                       ↓
        [0.021, -0.184, 0.731, ..., 0.092]
```

The individual numbers are not normally meaningful to us. What matters is that the resulting representation can be used to compare the semantic relationship between pieces of text.

For example, these sentences express similar ideas:

```text
"Customers can request a refund within 30 days."

"Refunds are available for customers for 30 days."
```

A good embedding model should represent their semantic relationship in a way that makes them useful for retrieval.

We will study embedding models and their trade-offs in **Module 5**.

---
## 6. Indexing

After generating embeddings, we need to make the information searchable.

A stored record might conceptually look like:

```python
{
    "chunk_id": "chunk_001",
    "document_id": "refund_policy",
    "text": "Customers can request a refund within 30 days.",
    "embedding": [...],
    "metadata": {
        "page": 4,
        "section": "Refund Eligibility"
    }
}
```

Notice that the vector is not the only thing we keep. We also preserve metadata that can later help with filtering, citations, provenance, and access control.

A vector database such as Qdrant can be used to store and search these vectors. We will study this in **Module 6**.

---
## 7. The Knowledge Base

After indexing, our raw documents have become a searchable knowledge representation:

```text
                  KNOWLEDGE BASE

        ┌──────────────────────────────┐
        │                              │
        │   Chunks                     │
        │   Embeddings                 │
        │   Metadata                   │
        │   Provenance                 │
        │                              │
        └──────────────────────────────┘
```

The indexing pipeline has prepared our knowledge. Now we can handle user questions.

---
# Part II — Answering a Question

Suppose a user asks:

> **Can I request a refund after 30 days?**

We don't want to send the entire knowledge base to the LLM. We first need to find the information relevant to this question.

---
## 8. Query Processing

The user's question may be processed before retrieval.

Possible techniques include:

- Query rewriting
- Query expansion
- Query decomposition
- Metadata extraction
- Intent detection

These techniques are not automatically necessary. One principle we'll follow throughout the course is:

> **Don't add complexity unless it solves a measurable problem.**

---
## 9. Retrieval

The retriever searches the knowledge base for information relevant to the user's question.

For semantic retrieval:

```text
User Query
    ↓
Query Embedding
    ↓
Vector Search
    ↓
Relevant Chunks
```

Imagine a knowledge base containing 100,000 chunks. We don't want to send all 100,000 to the LLM.

Instead:

```text
100,000 chunks
       ↓
    Retrieval
       ↓
   50 candidates
```

Those candidates can then be passed to another stage such as reranking.

---
## 10. Reranking

Initial retrieval is designed to efficiently find potentially relevant candidates. A reranker can then examine the relationship between the query and those candidates more carefully.

```text
Query
  │
  ▼
Initial Retrieval
  │
  ▼
50 Candidates
  │
  ▼
Reranker
  │
  ▼
5 Best Candidates
```

This creates a two-stage retrieval architecture:

```text
Fast retrieval
      ↓
Candidate selection
      ↓
More precise reranking
```

We will study reranking in **Module 9**.

---
## 11. Context Assembly

The retrieved information now needs to be provided to the LLM.

For example:

```text
SYSTEM

Answer using the supplied sources.
Do not invent information.

USER QUESTION

Can I request a refund after 30 days?

RETRIEVED CONTEXT

[Source 1]
Customers can request refunds within 30 days.

[Source 2]
Refund requests after 30 days are generally not accepted.

[Source 3]
Exceptions may apply in specific circumstances.
```

This combination of instructions, the user query, and retrieved information forms the context supplied to the model.

Context construction matters because too little information can omit necessary evidence, while too much irrelevant information can introduce noise.

We'll study this in **Module 10 — Context Engineering**.

---
## 12. Generation

The LLM receives the user question together with the retrieved context.

```text
User Query
     +
Retrieved Context
     ↓
    LLM
     ↓
Generated Answer
```

For example:

> Refund requests are generally accepted within 30 days. Requests after 30 days are normally not accepted, although specific exceptions may apply.

The important distinction is:

**The retriever finds information.**

**The LLM generates the response using that information.**

These are different problems and should be evaluated separately.

---
## 13. Citations and Provenance

A useful RAG system should ideally be able to trace generated information back to its source.

Instead of only producing:

> Refund requests are generally accepted within 30 days.

we may want:

> Refund requests are generally accepted within 30 days.
>
> **Source:** Refund Policy, page 4, "Refund Eligibility"

The provenance chain can look like:

```text
Answer
  ↓
Retrieved Chunk
  ↓
Document
  ↓
Page
  ↓
Original Source
```

This is why metadata needs to survive the earlier stages of the pipeline.

---
# Part III — The Two Pipelines Together

We can now combine everything we've learned:

```text
                    INDEXING TIME

Documents
    │
    ▼
Ingestion
    │
    ▼
Chunking
    │
    ▼
Embeddings
    │
    ▼
Indexing
    │
    ▼
Knowledge Base
    │
    │
    │                  QUERY TIME
    │
    │              User Query
    │                  │
    └──────────────► Retrieval
                       │
                       ▼
                   Reranking
                       │
                       ▼
                Context Assembly
                       │
                       ▼
                      LLM
                       │
                       ▼
                  Answer + Sources
```

The indexing pipeline prepares knowledge. The query pipeline uses that knowledge.

---
## 14. RAG Is More Than an LLM

A common misconception is that a RAG application is essentially:

```text
Documents → LLM → Answer
```

The actual system contains multiple stages:

```text
Ingestion
    ↓
Chunking
    ↓
Embedding
    ↓
Indexing
    ↓
Retrieval
    ↓
Reranking
    ↓
Context Assembly
    ↓
Generation
```

This means a poor answer does not necessarily mean the LLM is the problem.

For example:

```text
Poor ingestion
      ↓
Poor chunks
      ↓
Poor embeddings
      ↓
Poor retrieval
      ↓
Wrong context
      ↓
LLM
      ↓
Wrong answer
```

RAG engineering therefore requires us to understand and evaluate the entire pipeline.

---
## 15. Where Can a RAG System Fail?

| Stage | Example failure |
|---|---|
| Ingestion | Important text is not extracted |
| Chunking | Related information is separated |
| Embeddings | Similar concepts are poorly represented |
| Retrieval | Relevant chunks aren't returned |
| Reranking | Relevant candidates are incorrectly ranked |
| Context | Important information is omitted or buried in noise |
| Generation | The LLM misinterprets retrieved information |
| Provenance | Source information is lost |

This is why evaluating a RAG system requires more than asking whether the final answer looks good.

We need to understand **which component caused the result**.

---
# Part IV — From Simple RAG to Production RAG

The simplest RAG system might look like:

```text
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Vector Search
    ↓
LLM
    ↓
Answer
```

Our eventual system will be more sophisticated:

```text
                    Documents
                        ↓
              Robust Ingestion
                        ↓
             Structure-Aware Chunking
                        ↓
                   Embeddings
                        ↓
                 Vector Database
                        ↓
              ┌─────────┴─────────┐
              ↓                   ↓
         Dense Search           BM25
              ↓                   ↓
              └─────────┬─────────┘
                        ↓
                       RRF
                        ↓
                    Reranking
                        ↓
                Context Assembly
                        ↓
                       LLM
                        ↓
                Answer + Citations
                        ↓
                    Evaluation
```

We will build toward this architecture one component at a time.

---
## 16. What We'll Learn Next

Now that we understand the complete RAG architecture, we can zoom into each component:

```text
RAG Architecture
       ↓
Document Ingestion
       ↓
Chunking
       ↓
Embeddings
       ↓
Vector Databases
       ↓
Semantic Retrieval
       ↓
Hybrid Retrieval
       ↓
Reranking
       ↓
Context Engineering
       ↓
Grounded Generation
       ↓
Evaluation
       ↓
Production RAG
```

The next notebook will build a **minimal end-to-end RAG system**. It will deliberately be simple so that we can see the mechanics clearly before improving individual components.

---
# Key Takeaways

1. **RAG is an architecture, not a model.**
2. RAG has two major workflows: **indexing** and **querying**.
3. Indexing transforms documents into searchable representations.
4. Querying finds relevant information and supplies it to the LLM.
5. **Retrieval and generation are separate problems.**
6. The LLM is only one component of the system.
7. Metadata and provenance allow retrieved information to be traced back to its source.
8. A RAG system can fail at any stage, so we need to evaluate the pipeline as a system.

The central mental model is:

```text
Documents → Ingestion → Chunking → Embeddings → Index
                                                    ↓
Query → Retrieval → Reranking → Context → LLM → Answer
```
